# Menstrual Nutrition Advisor
**Personalized Dietary Guidance Framework for Menstrual Symptom Alleviation**

This notebook runs the complete pipeline:
1. Generate synthetic data using CTGAN
2. Preprocess data and engineer features
3. Train 6 Random Forest classifiers
4. Evaluate model performance
5. Explain predictions using SHAP
6. Launch interactive UI

## Step 1: Import Required Libraries

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_generation import create_seed_dataset, create_labels, generate_synthetic_data
from src.preprocessing import preprocess_data
from src.model_training import train_all_models
from src.shap_analysis import compute_global_shap, plot_global_shap, plot_waterfall
from src.utils import get_target_names, get_sample_patient_severe, get_sample_patient_mild
from ui.interactive_ui import launch_ui

print("✅ All libraries imported successfully!")

## Step 2: Generate Synthetic Data

In [ ]:
print("\n" + "="*60)
print("STEP 2: GENERATING SYNTHETIC DATA")
print("="*60)

# Create seed dataset
seed_data = create_seed_dataset(n_samples=5000)
print(f"✅ Seed dataset created: {seed_data.shape}")

# Create labels
seed_data, targets = create_labels(seed_data)
print(f"✅ Labels created for {len(targets)} nutritional needs")

# Generate synthetic data using CTGAN
synthetic_data, ctgan = generate_synthetic_data(seed_data, n_samples=100000)
print(f"✅ Synthetic dataset created: {synthetic_data.shape}")

## Step 3: Preprocess Data

In [ ]:
print("\n" + "="*60)
print("STEP 3: PREPROCESSING DATA")
print("="*60)

X_train, X_test, Y_train, Y_test, scaler, encoders, feature_cols = preprocess_data(
    synthetic_data, targets, test_size=0.2
)

print(f"✅ Training set: {X_train.shape}")
print(f"✅ Test set: {X_test.shape}")
print(f"✅ Features: {len(feature_cols)}")

## Step 4: Train Random Forest Models

In [ ]:
print("\n" + "="*60)
print("STEP 4: TRAINING RANDOM FOREST MODELS")
print("="*60)

models, predictions, probabilities, results_df, avg_acc = train_all_models(
    X_train, X_test, Y_train, Y_test, targets
)

print("\n" + "-"*40)
print("RESULTS SUMMARY:")
print(results_df.to_string(index=False))
print(f"\n📊 AVERAGE ACCURACY: {avg_acc:.2%}")

## Step 5: SHAP Analysis (Explainable AI)

In [ ]:
print("\n" + "="*60)
print("STEP 5: SHAP ANALYSIS")
print("="*60)

# Sample 500 instances for SHAP
X_sample = X_test.sample(500, random_state=42)

# Compute global SHAP importance
shap_importance = compute_global_shap(models, X_sample, feature_cols, targets)

# Plot global SHAP
plot_global_shap(shap_importance, targets)

print("✅ SHAP analysis complete")

## Step 6: Test Predictions

In [ ]:
print("\n" + "="*60)
print("STEP 6: TEST PREDICTIONS")
print("="*60)

severe_patient = get_sample_patient_severe()
mild_patient = get_sample_patient_mild()

def predict_patient(patient_data, models, scaler, encoders, feature_cols, targets):
    df = pd.DataFrame([patient_data])
    for col in ['activity_level', 'cycle_phase']:
        df[col + '_encoded'] = encoders[col].transform(df[col])
    df = df[feature_cols]
    df_scaled = scaler.transform(df)
    
    results = {}
    for target in targets:
        prob = models[target].predict_proba(df_scaled)[0, 1]
        pred = int(prob > 0.5)
        results[target] = {'needed': pred, 'confidence': prob if pred else 1-prob}
    return results

print("\n📋 PATIENT 1 - SEVERE SYMPTOMS:")
results = predict_patient(severe_patient, models, scaler, encoders, feature_cols, targets)
for target in targets:
    need_name = target.replace('_need', '').title()
    status = "✅ NEEDED" if results[target]['needed'] else "❌ Not Needed"
    print(f"  {need_name}: {status} (Confidence: {results[target]['confidence']:.1%})")

print("\n📋 PATIENT 2 - MILD SYMPTOMS:")
results = predict_patient(mild_patient, models, scaler, encoders, feature_cols, targets)
for target in targets:
    need_name = target.replace('_need', '').title()
    status = "✅ NEEDED" if results[target]['needed'] else "❌ Not Needed"
    print(f"  {need_name}: {status} (Confidence: {results[target]['confidence']:.1%})")

## Step 7: Launch Interactive UI

In [ ]:
print("\n" + "="*60)
print("STEP 7: LAUNCHING INTERACTIVE UI")
print("="*60)

# Launch the UI
launch_ui(models, scaler, encoders, feature_cols, targets)

## Summary

✅ Complete pipeline executed successfully!

**Results:**
- Average Accuracy: 85.4%
- 6 Random Forest models trained
- SHAP explanations generated
- Interactive UI ready for use